In [4]:
import torch
from torch import nn
import torch.nn.functional as F

In [5]:
embed_dim, attention_dim = 20, 5

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embed_dim, attention_dim):
        super().__init__()
        self.query = nn.Linear(embed_dim, attention_dim, bias = False)
        self.key = nn.Linear(embed_dim, attention_dim, bias = False)
        self.value = nn.Linear(embed_dim, attention_dim, bias = False)

    def forward(self, x):
        query = self.query(x)
        key = self.key(x)
        value = self.value(x)

        scores = torch.matmul(query, key.transpose(-2, -1))
        scores = scores / key.size(-1) ** 0.5 # smoothing

        attention_weights = F.softmax(scores, dim = -1)
        weighted_values = torch.matmul(attention_weights, value)

        return weighted_values

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        attention_dim = embed_dim // num_heads
        self.attentions = nn.ModuleList([SelfAttention(embed_dim, attention_dim) for _ in range(num_heads)])
        self.fc = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        head_outputs = []
        for attention in self.attentions:
            head_outputs.append(attention(x))

        concatenated_head = torch.cat(head_outputs, dim = -1)
        output = self.fc(concatenated_head)

        return output


In [8]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim, ff_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, n_head):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.multihead_attention = MultiheadAttention(embed_dim, n_head)

        self.layer_norm2 = nn.LayerNorm(embed_dim)
        self.feed_forward = FeedForward(embed_dim, embed_dim * 4)

    def forward(self, x):
        x = x + self.multihead_attenntion(self.layer_norm1(x))
        x = x + self.feed_forward(self.layer_norm2(x))
        return x